# 4. Small salary-cap-share model

Fit a compact Ridge regression to estimate next-season salary as a share of the salary cap. Features combine the shared model's projected production with prior-season age, role, and target season. Hometown, prior salary, and the test split are excluded.

Training production forecasts are generated out-of-fold so each salary example uses predictions from a profile network that did not train on that row. Compare the model with the existing historical-comp median on development data.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import sys
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

try:
    from IPython.display import display
except ImportError:
    display = print

rootPath = Path.cwd()
while not (rootPath / 'data' / 'processed').exists() and rootPath != rootPath.parent:
    rootPath = rootPath.parent
if not (rootPath / 'data' / 'processed').exists(): raise FileNotFoundError('Open this notebook from inside the project.')
sys.path.insert(0, str(rootPath))


processedPath = rootPath / 'data' / 'processed'
resultsPath = rootPath / 'results' / 'development' / 'multitask_profile'
modelPath = rootPath / 'models' / 'numpy_salary_ridge'
print('Project: ' + str(rootPath))

## Shared network and profile-target implementations

These implementations are included directly so this workflow does not import project Python modules.


In [ ]:
SKILL_TARGETS = [
    "pie",
    "ws_48",
    "bpm",
    "ts_percent",
    "usg_percent",
    "trb_percent",
    "ast_percent",
    "stl_percent",
    "blk_percent",
    "tov_percent",
    "x3p_ar",
    "f_tr",
    "e_fg_percent",
    "x3p_percent",
    "ft_percent",
]
PROFILE_TARGETS = SKILL_TARGETS + ["minutes"]
PCA_TARGETS = [
    target for target in SKILL_TARGETS
    if target not in {"x3p_percent", "ft_percent"}
]
NEXT_SEASON_COLUMNS = {
    target: ("next_season_minutes" if target == "minutes" else "next_season_" + target)
    for target in PROFILE_TARGETS
}
BASELINE_COLUMNS = {
    target: ("mp" if target == "minutes" else target)
    for target in PROFILE_TARGETS
}

In [ ]:
def initialize_parameters(layer_dims, seed=42):
    """Initialize a ReLU MLP with a multi-output linear final layer."""
    if len(layer_dims) < 2 or any(size < 1 for size in layer_dims):
        raise ValueError("Layer dimensions must be positive and include input/output sizes.")
    rng = np.random.default_rng(seed)
    parameters = {}
    for layer in range(1, len(layer_dims)):
        scale = np.sqrt(2.0 / layer_dims[layer - 1])
        parameters[f"W{layer}"] = (
            rng.standard_normal((layer_dims[layer], layer_dims[layer - 1])) * scale
        ).astype(np.float32)
        parameters[f"b{layer}"] = np.zeros((layer_dims[layer], 1), dtype=np.float32)
    return parameters

In [ ]:
def forward(inputs, parameters, keep_prob=1.0, training=False, rng=None):
    """Forward pass; input shape is features-by-examples."""
    if not 0 < keep_prob <= 1:
        raise ValueError("keep_prob must be in (0, 1].")
    if training and keep_prob < 1 and rng is None:
        raise ValueError("A random generator is required when dropout is enabled.")
    activations = inputs
    hidden_caches = []
    n_layers = len(parameters) // 2
    for layer in range(1, n_layers):
        previous = activations
        z = parameters[f"W{layer}"] @ previous + parameters[f"b{layer}"]
        activations = np.maximum(z, 0)
        dropout_mask = None
        if training and keep_prob < 1:
            dropout_mask = (
                (rng.random(activations.shape) < keep_prob).astype(np.float32)
                / keep_prob
            )
            activations = activations * dropout_mask
        hidden_caches.append((previous, z, dropout_mask))
    output_layer = n_layers
    output_input = activations
    predictions = (
        parameters[f"W{output_layer}"] @ output_input
        + parameters[f"b{output_layer}"]
    )
    return predictions, (hidden_caches, output_input)

In [ ]:
def masked_mse(predictions, targets, mask):
    """Mean of per-target MSEs, each averaged over its observed labels."""
    mask = np.asarray(mask, dtype=bool)
    counts = mask.sum(axis=1)
    valid_targets = counts > 0
    if not valid_targets.any():
        raise ValueError("Batch has no observed targets.")
    per_target = np.zeros(predictions.shape[0], dtype=np.float64)
    for target in np.flatnonzero(valid_targets):
        errors = predictions[target, mask[target]] - targets[target, mask[target]]
        per_target[target] = np.mean(errors.astype(np.float64) ** 2)
    return float(per_target[valid_targets].mean()), per_target, counts

In [ ]:
def backward(predictions, targets, mask, caches, parameters, weight_decay=0.0):
    """Backpropagate a task-balanced masked MSE and coupled L2 penalty."""
    hidden_caches, output_input = caches
    n_layers = len(parameters) // 2
    counts = mask.sum(axis=1).astype(np.float32)
    n_valid_targets = max(int(np.count_nonzero(counts)), 1)
    safe_counts = np.maximum(counts, 1.0)
    d_z = (
        2.0
        * (predictions - targets)
        * mask
        / safe_counts[:, None]
        / n_valid_targets
    )
    gradients = {}
    gradients[f"dW{n_layers}"] = (
        d_z @ output_input.T + weight_decay * parameters[f"W{n_layers}"]
    )
    gradients[f"db{n_layers}"] = (
        d_z.sum(axis=1, keepdims=True) + weight_decay * parameters[f"b{n_layers}"]
    )
    d_a = parameters[f"W{n_layers}"].T @ d_z

    for layer in reversed(range(1, n_layers)):
        previous, z, dropout_mask = hidden_caches[layer - 1]
        if dropout_mask is not None:
            d_a = d_a * dropout_mask
        d_z = d_a * (z > 0)
        gradients[f"dW{layer}"] = (
            d_z @ previous.T + weight_decay * parameters[f"W{layer}"]
        )
        gradients[f"db{layer}"] = (
            d_z.sum(axis=1, keepdims=True) + weight_decay * parameters[f"b{layer}"]
        )
        d_a = parameters[f"W{layer}"].T @ d_z
    return gradients

In [ ]:
def adam_initialize(parameters):
    first = {key: np.zeros_like(value) for key, value in parameters.items()}
    second = {key: np.zeros_like(value) for key, value in parameters.items()}
    return first, second

In [ ]:
def adam_update(parameters, gradients, first, second, step, learning_rate,
                beta1=0.9, beta2=0.999, epsilon=1e-8):
    for key in parameters:
        first[key] = beta1 * first[key] + (1.0 - beta1) * gradients["d" + key]
        second[key] = beta2 * second[key] + (1.0 - beta2) * gradients["d" + key] ** 2
        first_hat = first[key] / (1.0 - beta1**step)
        second_hat = second[key] / (1.0 - beta2**step)
        parameters[key] -= learning_rate * first_hat / (np.sqrt(second_hat) + epsilon)
    return parameters, first, second

In [ ]:
def predict(inputs, parameters, target_means, target_scales, minute_index=None):
    predictions, _ = forward(inputs, parameters)
    decoded = predictions.astype(np.float64).T * target_scales + target_means
    if minute_index is not None:
        decoded[:, minute_index] = np.maximum(decoded[:, minute_index], 0.0)
    return decoded

## Load training and development data

Only reliable, positive salary-cap-share labels are used. Source-review-flagged records are excluded from salary-model fitting and scoring.

In [ ]:
with np.load(processedPath / 'model_data_rates.npz', allow_pickle=False) as modelData:
    xTrain = modelData['X_train'].astype(np.float32)
    yTrainProfile = modelData['profile_y_train'].astype(np.float64)
    trainTargetMask = modelData['profile_mask_train'].astype(bool)
    featureNames = modelData['feature_names'].astype(str).tolist()
    targetNames = modelData['profile_target_names'].astype(str).tolist()

trainRows = pd.read_csv(processedPath / 'train_rows_rates.csv')
devRows = pd.read_csv(processedPath / 'dev_rows_rates.csv')
salaryPairs = pd.read_csv(processedPath / 'salary_cap_normalized_pairs.csv')
salaryColumns = ['player_id','target_season','salary_cap_share','salary_available_check','salary_needs_source_review','salary_cap_usd']
salaryFeatureRows = salaryPairs[salaryColumns].rename(columns={'salary_available_check':'model_salary_available','salary_needs_source_review':'model_salary_needs_review'})
trainRows = trainRows.merge(salaryFeatureRows, on=['player_id','target_season'], how='left', validate='one_to_one')
devRows = devRows.merge(salaryFeatureRows, on=['player_id','target_season'], how='left', validate='one_to_one')

reliableTrainMask = trainRows.model_salary_available.fillna(False).to_numpy() & ~trainRows.model_salary_needs_review.fillna(True).to_numpy() & trainRows.salary_cap_share.gt(0).fillna(False).to_numpy()
reliableDevMask = devRows.model_salary_available.fillna(False).to_numpy() & ~devRows.model_salary_needs_review.fillna(True).to_numpy() & devRows.salary_cap_share.gt(0).fillna(False).to_numpy()
print('Training rows with reliable salaries: ' + str(int(reliableTrainMask.sum())))
print('Development rows with reliable salaries: ' + str(int(reliableDevMask.sum())))
print('Rows flagged for salary review, including missing labels: ' + str(int(trainRows.model_salary_needs_review.fillna(True).sum())))
if targetNames != PROFILE_TARGETS: raise ValueError('Target order does not match the shared profile model.')
if len(trainRows) != len(xTrain): raise ValueError('Training rows do not align with model inputs.')

## Generate out-of-fold training forecasts

Use five deterministic folds. Each held-out training fold receives forecasts from a profile network trained on the other folds only. The run's selected profile settings are fixed at dropout 0.1 and 66 epochs. These folds are random, not time-ordered, so results are exploratory and need a chronological backtest before being treated as future-market evidence.

In [ ]:
foldCount = 5
hiddenWidths = [64, 32]
dropoutRate = 0.1
learningRate = 0.001
weightDecay = 0.0001
batchSize = 64
trainingEpochs = 66
layerWidths = [xTrain.shape[1], *hiddenWidths, len(PROFILE_TARGETS)]
foldOrder = np.random.default_rng(42).permutation(len(xTrain))
foldLabels = np.empty(len(xTrain), dtype=int)
foldLabels[foldOrder] = np.arange(len(xTrain)) % foldCount
oofForecasts = np.full((len(xTrain), len(PROFILE_TARGETS)), np.nan, dtype=np.float64)

for foldNumber in range(foldCount):
    fitMask = foldLabels != foldNumber
    holdoutMask = foldLabels == foldNumber
    xFit = xTrain[fitMask]
    yFit = yTrainProfile[fitMask]
    maskFit = trainTargetMask[fitMask]
    targetMeans = np.zeros(len(PROFILE_TARGETS), dtype=np.float64)
    targetScales = np.ones(len(PROFILE_TARGETS), dtype=np.float64)
    scaledTargets = np.zeros(yFit.shape, dtype=np.float32)

    for targetIndex in range(len(PROFILE_TARGETS)):
        observedTargets = yFit[maskFit[:, targetIndex], targetIndex]
        if not len(observedTargets): raise ValueError('A fold has no labels for ' + PROFILE_TARGETS[targetIndex])
        targetMeans[targetIndex] = observedTargets.mean()
        targetScales[targetIndex] = observedTargets.std()
        if targetScales[targetIndex] <= 0: raise ValueError('A fold has no target variation for ' + PROFILE_TARGETS[targetIndex])
        targetMask = maskFit[:, targetIndex]
        scaledTargets[targetMask, targetIndex] = ((yFit[targetMask, targetIndex] - targetMeans[targetIndex]) / targetScales[targetIndex]).astype(np.float32)

    parameters = initialize_parameters(layerWidths, seed=42 + foldNumber)
    firstMoment, secondMoment = adam_initialize(parameters)
    shuffleRandom = np.random.default_rng(52 + foldNumber)
    dropoutRandom = np.random.default_rng(62 + foldNumber)
    updateStep = 0

    for epochNumber in range(trainingEpochs):
        batchOrder = shuffleRandom.permutation(len(xFit))
        for batchStart in range(0, len(batchOrder), batchSize):
            batchIndices = batchOrder[batchStart:batchStart + batchSize]
            batchOutput, cache = forward(xFit[batchIndices].T, parameters, keep_prob=1 - dropoutRate, training=True, rng=dropoutRandom)
            gradients = backward(batchOutput, scaledTargets[batchIndices].T, maskFit[batchIndices].T.astype(np.float32), cache, parameters, weightDecay)
            updateStep += 1
            parameters, firstMoment, secondMoment = adam_update(parameters, gradients, firstMoment, secondMoment, updateStep, learningRate)

    oofForecasts[holdoutMask] = predict(xTrain[holdoutMask].T, parameters, targetMeans, targetScales, minute_index=PROFILE_TARGETS.index('minutes'))
    print('Finished fold ' + str(foldNumber + 1) + ' / ' + str(foldCount))

if not np.isfinite(oofForecasts).all(): raise ValueError('Out-of-fold forecasts contain missing values.')
print('Out-of-fold forecast matrix: ' + str(oofForecasts.shape))

## Build model inputs and fit Ridge regression

Numeric inputs are standardized, position is one-hot encoded, and the salary-cap share target is modeled on a log scale. Prior salary is intentionally excluded.

In [ ]:
def buildSalaryFeatures(playerRows, forecastRows):
    salaryFeatures = playerRows[['age','pos','g','gs','mp','mp_per_game','target_season']].copy()
    salaryFeatures['pos'] = salaryFeatures.pos.fillna('Unknown').astype(str)
    forecastFrame = pd.DataFrame(forecastRows, columns=PROFILE_TARGETS, index=salaryFeatures.index)
    forecastFrame = forecastFrame.rename(columns={target:'forecast_' + target for target in PROFILE_TARGETS})
    return pd.concat([salaryFeatures, forecastFrame], axis=1)

trainSalaryFeatures = buildSalaryFeatures(trainRows, oofForecasts)
trainSalaryTargets = trainRows.salary_cap_share.to_numpy(dtype=float)
numericColumns = ['age','g','gs','mp','mp_per_game','target_season'] + ['forecast_' + target for target in PROFILE_TARGETS]
categoricalColumns = ['pos']
featureTransform = ColumnTransformer([
    ('numeric', StandardScaler(), numericColumns),
    ('position', OneHotEncoder(handle_unknown='ignore'), categoricalColumns),
])
salaryModel = Pipeline([
    ('features', featureTransform),
    ('ridge', Ridge(alpha=10.0)),
])
salaryModel.fit(trainSalaryFeatures.loc[reliableTrainMask], np.log1p(trainSalaryTargets[reliableTrainMask]))
print('Fitted Ridge salary model on ' + str(int(reliableTrainMask.sum())) + ' reliable salary rows.')

## Compare against comp median on development data

The comparison cohort contains development players with reliable salary labels and an available comp estimate. Metrics are in salary-cap percentage points.

In [ ]:
runFolders = []
for resultFolder in resultsPath.iterdir():
    metadataFile = rootPath / 'models' / 'numpy_multitask_profile' / resultFolder.name / 'metadata.json'
    if resultFolder.is_dir() and metadataFile.exists():
        modelMetadata = json.loads(metadataFile.read_text(encoding='utf-8'))
        if int(modelMetadata.get('training_minimum_prior_games', 0)) == 0: runFolders.append(resultFolder)
if not runFolders: raise FileNotFoundError('No unrestricted profile-model result was found.')
profileRun = max(runFolders, key=lambda path: path.stat().st_mtime)
compEstimates = pd.read_csv(profileRun / 'development_comp_salary_estimates.csv')
devPredictionRows = pd.read_csv(profileRun / 'development_profile_predictions.csv')
if not devPredictionRows[['player_id','season','target_season']].equals(devRows[['player_id','season','target_season']]): raise ValueError('Development prediction rows are misaligned.')
devFeatures = buildSalaryFeatures(devRows, devPredictionRows[['predicted_' + target for target in PROFILE_TARGETS]].to_numpy())
devModelEstimate = np.maximum(0, np.expm1(salaryModel.predict(devFeatures)))
devResults = devRows[['player','player_id','season','target_season','salary_cap_share','salary_cap_usd','model_salary_available','model_salary_needs_review']].copy()
devResults['salaryModelSharePct'] = devModelEstimate * 100
compEstimates = compEstimates.rename(columns={'query_player_id':'player_id','query_target_season':'target_season'})
devResults = devResults.merge(compEstimates[['player_id','target_season','estimated_salary_cap_share_pct','salary_estimate_available']], on=['player_id','target_season'], how='left', validate='one_to_one')
devResults['compMedianSharePct'] = devResults.estimated_salary_cap_share_pct
devResults['salaryModelDollars'] = devResults.salaryModelSharePct / 100 * devResults.salary_cap_usd
devResults['compMedianDollars'] = devResults.compMedianSharePct / 100 * devResults.salary_cap_usd
comparisonMask = reliableDevMask & devResults.salary_estimate_available.fillna(False).to_numpy()
if not comparisonMask.any(): raise ValueError('No eligible development rows for salary-model comparison.')

def salaryMetrics(actualShare, predictedShare):
    error = (predictedShare - actualShare) * 100
    return {
        'rows': int(len(error)),
        'mae_cap_percentage_points': float(np.mean(np.abs(error))),
        'median_absolute_error_cap_points': float(np.median(np.abs(error))),
        'rmse_cap_percentage_points': float(np.sqrt(np.mean(error ** 2))),
    }

actualShares = devResults.loc[comparisonMask, 'salary_cap_share'].to_numpy(dtype=float)
modelShares = devResults.loc[comparisonMask, 'salaryModelSharePct'].to_numpy(dtype=float) / 100
compShares = devResults.loc[comparisonMask, 'compMedianSharePct'].to_numpy(dtype=float) / 100
comparisonTable = pd.DataFrame([
    {'method':'Ridge salary model', **salaryMetrics(actualShares, modelShares)},
    {'method':'Nearest-comp median', **salaryMetrics(actualShares, compShares)},
])
display(comparisonTable.round(3))
print('Reliable development comparison rows: ' + str(int(comparisonMask.sum())))
print('Test split loaded: False')

## Inspect examples and save the model

Salary-model and comp estimates are both market references. Review individual differences and the error comparison before choosing which estimate to present.

In [ ]:
modelRunId = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
salaryModelFolder = modelPath / modelRunId
salaryModelFolder.mkdir(parents=True, exist_ok=False)
joblib.dump(salaryModel, salaryModelFolder / 'salary_ridge.joblib')
modelDetails = {
    'model':'Ridge regression on log salary-cap share',
    'target':'next-season salary_cap_share',
    'training_rows':int(reliableTrainMask.sum()),
    'development_comparison_rows':int(comparisonMask.sum()),
    'profile_forecasts':'five-fold out-of-fold predictions for training; saved unrestricted profile forecasts for development',
    'numeric_features':numericColumns,
    'categorical_features':categoricalColumns,
    'excluded_features':['hometown','prior salary','player ID','team'],
    'ridge_alpha':10.0,
    'salary_source_policy':'positive available salary and not flagged for source review',
    'test_split_loaded':False,
    'development_results':comparisonTable.to_dict(orient='records'),
}
(salaryModelFolder / 'metadata.json').write_text(json.dumps(modelDetails, indent=2) + '\n', encoding='utf-8')
resultFolder = resultsPath / profileRun.name
devResults.to_csv(resultFolder / 'development_salary_model_predictions.csv', index=False)
comparisonTable.to_csv(resultFolder / 'development_salary_model_comparison.csv', index=False)

exampleRows = devResults.loc[comparisonMask].copy()
exampleRows['absoluteModelError'] = (exampleRows.salaryModelSharePct - exampleRows.salary_cap_share * 100).abs()
exampleRows['absoluteCompError'] = (exampleRows.compMedianSharePct - exampleRows.salary_cap_share * 100).abs()
display(exampleRows.sort_values('absoluteModelError').head(10)[['player','target_season','salary_cap_share','salaryModelSharePct','compMedianSharePct','salaryModelDollars','compMedianDollars']])
print('Saved salary model: ' + str(salaryModelFolder))
print('Saved development comparison: ' + str(resultFolder / 'development_salary_model_comparison.csv'))

## Visual comparison
Each point compares an estimate with the observed development salary-cap share. Selected players are labeled; the diagonal is perfect agreement.

In [ ]:
import matplotlib.pyplot as plt

salaryFigure, salaryAxes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
actualSharePct = exampleRows.salary_cap_share.to_numpy() * 100
modelSharePct = exampleRows.salaryModelSharePct.to_numpy()
compSharePct = exampleRows.compMedianSharePct.to_numpy()
plotMaximum = max(40, float(np.quantile(np.concatenate([actualSharePct, modelSharePct, compSharePct]), 0.99)))
playerLabels = [('Isaiah Jackson', 2025), ('LeBron James', 2020), ('Jayson Tatum', 2025)]

for axis, estimateColumn, panelTitle, color in [
    (salaryAxes[0], 'salaryModelSharePct', 'Ridge salary model', '#3a86ff'),
    (salaryAxes[1], 'compMedianSharePct', 'Nearest-comp median', '#6a994e'),
]:
    axis.scatter(exampleRows.salary_cap_share * 100, exampleRows[estimateColumn], alpha=0.22, s=18, color=color)
    axis.plot([0, plotMaximum], [0, plotMaximum], color='#d1495b', linestyle='--')
    for playerName, seasonYear in playerLabels:
        playerRow = exampleRows[(exampleRows.player == playerName) & (exampleRows.target_season == seasonYear)]
        if not playerRow.empty:
            playerRow = playerRow.iloc[0]
            axis.annotate(playerName, (playerRow.salary_cap_share * 100, playerRow[estimateColumn]), xytext=(5, 5), textcoords='offset points', fontsize=8)
    axis.set(title=panelTitle, xlabel='Observed salary-cap share (%)', xlim=(0, plotMaximum), ylim=(0, plotMaximum))
salaryAxes[0].set_ylabel('Estimated salary-cap share (%)')
salaryFigure.suptitle('Development salary estimates vs observed salary-cap share')
salaryFigure.tight_layout()
salaryFigure.savefig(resultFolder / 'salary_model_comparison.png', dpi=160)
display(salaryFigure)
print('Saved chart: ' + str(resultFolder / 'salary_model_comparison.png'))

## Interpretation
- This is a production-based estimate of market salary, not a contract guarantee.
- The development comparison is exploratory. No test split was loaded or scored.
- Compare the Ridge model with the comp median before deciding which estimate to present.